## Execução local
Selecione o kernel **Python (FASE7)** e execute em ordem. Consulte `README_LOCAL.md` para instalação. Redes neurais usam PyTorch/CUDA; operações NumPy, áudio e modelos scikit-learn usam CPU.

In [ ]:
from pathlib import Path
import os, sys
# Localiza FASE7 tanto a partir do projeto quanto da pasta do notebook.
base = Path.cwd().resolve()
FASE7 = next((p for a in (base, *base.parents) for p in (a, a / "FASE7", a / "ANO1" / "FASE7") if p.name == "FASE7" and (p / "RNN").is_dir()), None)
if FASE7 is None:
    raise FileNotFoundError("Abra o Jupyter dentro da pasta FIAP ou FASE7.")
sys.path.insert(0, str(FASE7))
os.chdir(FASE7 / 'Cap 02')
print("Pasta de trabalho:", Path.cwd())
from fase7_torch import DEVICE
print("Dispositivo:", DEVICE)


FIAP

# TIAO - Redes Neurais

# Aula 4b - Série Temporal Multivariada

Predição baseada em múltiplas variáveis, para prever nível de poluição do ar.

In [ ]:
from pathlib import Path
if not Path("raw.csv").exists():
    from urllib.request import urlretrieve
    urlretrieve("https://raw.githubusercontent.com/jbrownlee/Datasets/master/pollution.csv", "raw.csv")


In [ ]:
import pandas as pd
raw = pd.read_csv("raw.csv")
dates = pd.to_datetime(raw[["year", "month", "day", "hour"]])
dataset = raw.drop(columns=["No", "year", "month", "day", "hour"])
dataset.index = dates
dataset.index.name = "date"
dataset.columns = ["pollution", "dew", "temp", "press", "wnd_dir", "wnd_spd", "snow", "rain"]
dataset["pollution"] = dataset["pollution"].fillna(0)
dataset = dataset.iloc[24:]
dataset.to_csv("pollution.csv")


In [ ]:
# Mostra as primeiras 10 linhas
dataset.head(10)

In [ ]:
from pandas import read_csv
from matplotlib import pyplot

# Carrega o dataset
dataset = read_csv('pollution.csv', header=0, index_col=0)
values = dataset.values

# Especifica as colunas para plotar
groups = [0, 1, 2, 3, 5, 6, 7]
i = 1
pyplot.figure()
for group in groups:
	pyplot.subplot(len(groups), 1, i)
	pyplot.plot(values[:, group])
	pyplot.title(dataset.columns[group], y=0.5, loc='right')
	i += 1
pyplot.show()

In [ ]:
import pandas as pd
import torch
from torch import nn
from fase7_torch import RegressorLSTM, treinar, prever
# Imports necessários
from math import sqrt
from numpy import concatenate
from matplotlib import pyplot
from pandas import read_csv
from pandas import DataFrame
from pandas import concat
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

# Converte a série em aprendizado supervisionado
def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):
	n_vars = 1 if type(data) is list else data.shape[1]
	df = DataFrame(data)
	cols, names = list(), list()
	# input sequence (t-n, ... t-1)
	for i in range(n_in, 0, -1):
		cols.append(df.shift(i))
		names += [('var%d(t-%d)' % (j+1, i)) for j in range(n_vars)]
	# forecast sequence (t, t+1, ... t+n)
	for i in range(0, n_out):
		cols.append(df.shift(-i))
		if i == 0:
			names += [('var%d(t)' % (j+1)) for j in range(n_vars)]
		else:
			names += [('var%d(t+%d)' % (j+1, i)) for j in range(n_vars)]
	# put it all together
	agg = concat(cols, axis=1)
	agg.columns = names
	# drop rows with NaN values
	if dropnan:
		agg.dropna(inplace=True)
	return agg



# Carrega a base de dados e exclui a coluna de direção do vento (categórica)
dataset = read_csv('pollution.csv', header=0, index_col=0)
dataset.drop(columns=['wnd_dir'], inplace=True)

# Garanta que todos os dados são numéricos
values = dataset.values
values = values.astype('float32')

# Normaliza as features
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(values[:365 * 24 + 3])  # Ajuste somente no período de treino.
scaled = scaler.transform(values)
dataset_scaled = pd.DataFrame(scaled, columns=dataset.columns)

# Escolhe a variável a ser prevista
target_variable = 'pollution'

# Separa X (features) e y (target)
X = dataset_scaled.drop(columns=[target_variable])  # Remove a variável alvo das entradas
y = dataset_scaled[[target_variable]] 							# Mantém a variável alvo separada

# Aplica a função para transformar os dados, usando 3 valores (lags) como entrada
n_lags = 3
df_X_supervised = series_to_supervised(X, n_in=n_lags, n_out=0)  # Somente entradas (X)
df_y_supervised = series_to_supervised(y, n_in=n_lags, n_out=1)  # Apenas saída (y)

# Juntando entradas (X) e saída (y)
df_final = pd.concat([df_X_supervised, df_y_supervised.iloc[:, -1]], axis=1)

# Exibindo o DataFrame final com X e y prontos para treinamento
display(df_final.head())

# Dividindo os dados em treino e teste
values = df_final.values
n_train_hours = 365 * 24
train = values[:n_train_hours, :]
test = values[n_train_hours:, :]

# Separando features e labels
train_X, train_y = train[:, :-1], train[:, -1]
test_X, test_y = test[:, :-1], test[:, -1]

# Reshape num valor 3D [samples, timesteps, features]
train_X = train_X.reshape((train_X.shape[0],
                           n_lags,
                           int(train_X.shape[1]/n_lags)))
test_X = test_X.reshape((test_X.shape[0],
                         n_lags,
                         int(test_X.shape[1]/n_lags)))
print()
display(train_X.shape, train_y.shape, test_X.shape, test_y.shape)

In [ ]:
model = RegressorLSTM(train_X.shape[2], hidden=50).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters())
history = treinar(model, train_X, train_y, optimizer, nn.L1Loss(),
                  epochs=50, batch_size=72, shuffle=False)
pyplot.plot(history["loss"], label="train")
pyplot.legend()
pyplot.show()
yhat = prever(model, test_X)
rmse = sqrt(mean_squared_error(test_y, yhat))
print(f"Teste RMSE (escala normalizada): {rmse:.3f}")
# Retorna a unidade original da concentração de poluentes.
y_real = (test_y - scaler.min_[0]) / scaler.scale_[0]
y_pred = (yhat[:, 0] - scaler.min_[0]) / scaler.scale_[0]
print(f"Teste RMSE (unidade original): {sqrt(mean_squared_error(y_real, y_pred)):.3f}")
pyplot.plot(y_pred[:100], label="y predito")
pyplot.plot(y_real[:100], label="y real")
pyplot.legend()
pyplot.show()
